# Project Deep Dive / Behavioral Round

The project deep dive is where you prove you've actually shipped ML systems. Interviewers aren't listening for fancy algorithms — they're listening for ownership, judgment under uncertainty, and the ability to learn from failure. This note gives you the framework and two complete example narratives.

## What Interviewers Test
- Can you lead an interviewer through a technical project without losing them?
- Do you know *why* you made each technical decision (not just what)?
- Can you quantify impact beyond "improved model performance"?
- How do you handle failure, setbacks, and ambiguity?
- Do you show ownership of the full system, not just your component?

## The STAR-ML Framework

STAR (Situation → Task → Action → Result) adapted for ML:

```
SITUATION   → Business context and why the problem mattered
TASK        → Your specific role and what success looked like
ACTIONS     → Technical decisions with explicit tradeoffs
              (what you tried, what failed, what you chose and why)
RESULTS     → Quantified impact at business and technical level
REFLECTION  → What you'd do differently (shows growth mindset)
```

**Timing in a 30-min round:**
- Situation + Task: 3–5 minutes (interviewer gets context)
- Actions: 15–20 minutes (the interview lives here)
- Results: 2–3 minutes (quantified, concrete)
- Reflection + follow-ups: 5 minutes


## Example Narrative 1: Improving a Recommendation Model

**Situation (1–2 min):**
"I was on the recommendations team at [Company]. Our homepage recommendation CTR was 4.2%, below the product target of 5%, and we were seeing that new users had particularly low engagement — our collaborative filtering model needed 10+ interactions before giving good recs."

**Task:**
"My task was to improve cold-start performance without sacrificing quality for established users. Success meant: no regression in CTR for existing users, and ≥30% improvement in 7-day retention for new users."

**Actions (detailed):**

*First attempt: content-based features*
"I started by adding item content features (category, price bucket, description embeddings from a frozen BERT). Training was fast. But offline NDCG only improved 0.2%. Post-mortem: content features were already implicitly encoded in collaborative signal for established users — I was adding noise, not signal."

*What actually worked: two-tower with content for cold-start*
"I pivoted to a two-tower architecture where new users used only content-side features (signup categories, device), while established users used collaborative embeddings. I trained them jointly with a shared item tower. This let me use the same serving infrastructure but switch user tower inputs based on interaction count."

*Deployment challenge:*
"The two-tower model was 3× larger than the old model and pushed us over our serving latency SLA. I profiled and found that item embedding lookup was the bottleneck. Solved it by pre-computing and caching item embeddings in Redis, updated every 30 minutes."

**Results:**
"New user 7-day retention: +38% (target was 30%). CTR for established users: unchanged (+0.1%, within guardrail). Latency p99: 45ms (SLA: 50ms, previous model: 30ms — accepted by product as worth the trade). Shipped to 100% of traffic after 2-week canary."

**Reflection:**
"I'd have done the offline analysis on content feature importance earlier — I spent two weeks on the content-only approach that wasn't going anywhere. A 48-hour feature ablation study before building would have saved time."


## Example Narrative 2: MLOps — Building a Retraining Pipeline

**Situation:**
"Our fraud detection model was deployed as a static artifact trained every quarter. After a major account takeover campaign in Q3, our model's precision dropped from 85% to 71% over 3 weeks because it hadn't seen the new attack patterns."

**Task:**
"Build an automated retraining pipeline that could re-deploy a new model within 72 hours of detecting significant precision degradation. My role was end-to-end: pipeline design, implementation, and the eval gate."

**Actions:**

*Pipeline architecture:*
"I built a DAG in Airflow: data extraction from the feature store → training job on Kubernetes → eval against a fixed holdout set → automatic promotion if metrics pass → canary deploy. The holdout set was key — it included labeled examples from the last attack campaign so we'd catch regression."

*The eval gate design:*
"I defined two gates: (1) AUC on the holdout must be ≥ current production model; (2) precision@10K (our review queue size) must be ≥ 80%. If both pass, the model auto-promotes to a 10% canary. If either fails, the pipeline fires a PagerDuty alert and blocks promotion."

*Failure that shaped the design:*
"In our first test run, a data pipeline bug caused 15% of training labels to be dropped silently. The model trained fine but AUC looked artificially high because it was evaluated on a less diverse holdout. I added data quality gates before training: label count must be within 10% of expected; positive rate must be within 20% of historical. The pipeline won't proceed if data gates fail."

**Results:**
"Time-to-redeploy on new attack patterns: from 6–8 weeks to 68 hours (first end-to-end run). Precision during the next attack campaign (Q1 next year): held at 83% vs previous quarter's 71% drop. Side effect: we discovered a mislabeled data issue that had been in production for 4 months because the pipeline exposed it."

**Reflection:**
"I underestimated the importance of data quality gates and learned them the hard way. Now I think about data quality checks before model quality checks — garbage in means the model metrics are meaningless."


## How to Handle Hard Follow-Up Questions

| Follow-up | Strong answer structure |
|---|---|
| "What was the hardest part?" | Specific technical challenge + why it was hard + how you resolved it |
| "What would you do differently?" | Honest reflection + what you'd change + what you learned |
| "How would you scale this 10x?" | Identify bottleneck → specific architectural change → expected result |
| "How did you validate that X worked?" | Offline eval metric + online A/B result + business metric |
| "What would break first if traffic doubled?" | Name the specific component + why + how you'd address it |

**Weak answers to avoid:**
- "It all went pretty smoothly" (no one believes this)
- "I'd use a different algorithm" without explaining what it would fix
- "We didn't have time to..." (shows lack of ownership)
- Passive language: "The model was trained by..." (own your work)


## Common Interview Questions

**Q: How should I pick which project to present?**
Pick the one where you have the most depth — where you can answer "why" for every decision, know what failed, and can quantify impact. This is usually the project you owned end-to-end, not the one with the most impressive tech. Recency matters less than depth.

**Q: What if my project didn't go well or had modest results?**
Present it honestly. "We tried X, it didn't work because Y, we pivoted to Z" shows learning and intellectual honesty. Interviewers know most ML projects don't get 10× improvements. A mediocre result with excellent explanation of why is more credible than an excellent result you can't explain.

**Q: How technical should I go?**
Match the room. Start at a level that a senior engineer would understand. If they probe technically ("can you explain the backpropagation through the two-tower?"), go deeper. If they redirect toward business impact, follow them. The best candidates can work at any level of abstraction on demand.

**Q: How do I handle "what impact did you have?" if I worked on a team?**
Use "I" not "we" for your specific contributions, while acknowledging the team context. "The team's overall impact was 3% CTR improvement. My specific contribution was the cold-start solution, which drove 1.2% of that." Don't over-claim teammates' work — interviewers cross-reference with your collaborators.

## Key Takeaways
- STAR-ML: Situation → Task → Actions (with tradeoffs) → Results → Reflection
- Lead with business problem, not algorithm — interviewers care why you built it, not just what
- Own failures: "it didn't work because X and here's what I learned" beats hiding failures
- Quantify everything: retention %, latency ms, precision %, throughput req/s, deploy time hours
- Know the full system, not just your component — be able to explain decisions you didn't make
- "What would you do differently?" is a gift — it's your chance to show growth mindset